# 06 — Feature Engineering and Model Testing

This notebook prepares the modeling dataset for time-series forecasting and tests different feature sets.

The objective is to transform the consolidated master dataset into a model-ready structure by:

- Filtering the relevant market / aggregate level
- Creating lag features from historical registrations
- Adding rolling averages to capture recent market momentum
- Encoding seasonality using month-based features
- Testing macroeconomic indicators with different lag assumptions
- Evaluating candidate feature sets through time-based validation

The output of this notebook is a clean feature matrix and a set of tested feature configurations that support the final model selection.

## Feature Design Principles

The feature engineering approach focuses on:
- capturing temporal dependencies through lagged variables
- incorporating macroeconomic context
- preserving interpretability of model inputs

In [ ]:
%pip install statsmodels

In [ ]:
%pip install matplotlib

In [ ]:
import pandas as pd
import numpy as np
from functools import reduce
import os

from statsmodels.tsa.stattools import adfuller


In [ ]:
# =========================================================
# Load datasets
# =========================================================
master_df_impute = pd.read_parquet("data/master_df_impute.parquet")

In [ ]:
master_df_impute.columns.tolist()

In [ ]:
# =========================================================
# Defyning target 
# =========================================================

target = "acea_total_registrations"

In [ ]:
# =========================================================
# 📊 Correlation with target (quick signal check)
# =========================================================

corr = master_df_impute_eu.corr(numeric_only=True)[target].sort_values(ascending=False)

In [ ]:
display(corr)

In [ ]:
# =========================================================
# 📊 Correlation with target (quick signal check)
# =========================================================

import pandas as pd

def lag_corr(df, feature, target, lags=13):
    results = []
    for lag in range(0, lags+1):
        corr = df[target].corr(df[feature].shift(lag))
        results.append((lag, corr))
    return pd.DataFrame(results, columns=["lag", "correlation"])

In [ ]:
# =========================================================
# Creating lags function for correlation
# =========================================================

def lag_corr(df, feature, target, lags=13):
    results = []
    
    for lag in range(0, lags + 1):
        corr = df[target].corr(df[feature].shift(lag))
        results.append(corr)
    
    return results

In [ ]:
macro_vars = [
    "consumer_confidence",
    "gdp_growth",
    "unemployment_rate",
    "interest_rate",
    "petrol_euro95",
    "petrol_euro95_yoy",
    "real_final_consumption",
    "cpi",
    "vehicle_parc",
    "vehicle_parc_yoy",
    "vehicle_production",
    "ev_adoption_ratio"
]

In [ ]:
# =========================================================
# 📊 Lagged correlation table
# =========================================================

lags = 13

lag_corr_dict = {}

for var in macro_vars:
    lag_corr_dict[var] = lag_corr(master_df_impute_eu, var, target, lags=lags)

lag_corr_table = pd.DataFrame.from_dict(
    lag_corr_dict,
    orient="index",
    columns=[f"lag_{i}" for i in range(0, lags + 1)]
)

lag_corr_table

In [ ]:
# Select only lag columns
lag_cols = [col for col in lag_corr_table.columns if "lag_" in col]

# Compute best lag
lag_corr_table["best_lag"] = lag_corr_table[lag_cols].abs().idxmax(axis=1)
lag_corr_table["max_corr"] = lag_corr_table[lag_cols].abs().max(axis=1)

# Sort
lag_corr_table = lag_corr_table.sort_values(by="max_corr", ascending=False)

lag_corr_table

In [ ]:
import matplotlib.pyplot as plt

for var in macro_vars:
    correlations = lag_corr(master_df_impute_eu, var, target)
    
    plt.plot(range(0, 14), correlations, label=var)

plt.axhline(0, linestyle="--")
plt.xlabel("Lag (months)")
plt.ylabel("Correlation")
plt.title("Lagged Correlation with Target")
plt.legend()
plt.show()

# Creating clean dataframes with the selected features for each model (based on the lagged correlations) 

In [ ]:
# =========================================================
# 1. Define target and selected lags
# =========================================================

target = "acea_total_registrations"

selected_lags_eu_macro = {
    "acea_total_registrations": 12,
    "interest_rate": 0,
    "gdp_growth": 1,
    "petrol_euro95": 4,
    "unemployment_rate": 0,
    "consumer_confidence": 1,
    "real_final_consumption": 7,
    "cpi": 0
}

selected_lags_eu_ml = {
    "acea_total_registrations": 12,
    "interest_rate": 0,
    "gdp_growth": 1,
    "petrol_euro95": 4,
    "unemployment_rate": 0,
    "consumer_confidence": 1,
    "real_final_consumption": 7,
    "cpi": 0,
    "ev_adoption_ratio": 7,
    "vehicle_production": 1,
    "vehicle_parc": 0
}

In [ ]:
# =========================================================
# 2. Create clean working copy
# =========================================================

model_df = master_df_impute_eu.copy()

# Ensure correct date sorting
model_df["date"] = pd.to_datetime(model_df["date"])
model_df = model_df.sort_values("date").reset_index(drop=True)

In [ ]:
# =========================================================
# 3. Baseline features
# =========================================================

baseline_lags = [1, 3, 6, 12]

for lag in baseline_lags:
    model_df[f"{target}_lag_{lag}"] = model_df[target].shift(lag)

# Optional rolling features for baseline
model_df[f"{target}_roll_mean_3"] = model_df[target].shift(1).rolling(3).mean()
model_df[f"{target}_roll_mean_6"] = model_df[target].shift(1).rolling(6).mean()

# Seasonality encoding
import numpy as np

model_df["month_sin"] = np.sin(2 * np.pi * model_df["month"] / 12)
model_df["month_cos"] = np.cos(2 * np.pi * model_df["month"] / 12)

In [ ]:
# =========================================================
# 4. Create macro lag features
# =========================================================

for feature, lag in selected_lags_eu_macro.items():
    if lag == 0:
        model_df[f"{feature}_lag_0"] = model_df[feature]
    else:
        model_df[f"{feature}_lag_{lag}"] = model_df[feature].shift(lag)

In [ ]:
# =========================================================
# 5. Create full-model lag features
# =========================================================

for feature, lag in selected_lags_eu_ml.items():
    if lag == 0:
        model_df[f"{feature}_lag_0"] = model_df[feature]
    else:
        model_df[f"{feature}_lag_{lag}"] = model_df[feature].shift(lag)

In [ ]:
# =========================================================
# 6. Define feature lists
# =========================================================

baseline_features = [
    f"{target}_lag_1",
    f"{target}_lag_3",
    f"{target}_lag_6",
    f"{target}_lag_12",
    f"{target}_roll_mean_3",
    f"{target}_roll_mean_6",
    "month_sin",
    "month_cos"
]

macro_features = baseline_features + [
    f"{feature}_lag_{lag}" for feature, lag in selected_lags_eu_macro.items()
]

full_model_features = baseline_features + [
    f"{feature}_lag_{lag}" for feature, lag in selected_lags_eu_ml.items()
]

In [ ]:
# =========================================================
# 7. Create final model-ready dataframes
# =========================================================

baseline_df = model_df[["date", target] + baseline_features].dropna().reset_index(drop=True)

macro_df = model_df[["date", target] + macro_features].dropna().reset_index(drop=True)

full_model_df = model_df[["date", target] + full_model_features].dropna().reset_index(drop=True)

In [ ]:
# =========================================================
# 8. Quick validation
# =========================================================

print("baseline_df shape:", baseline_df.shape)
print("macro_df shape:", macro_df.shape)
print("full_model_df shape:", full_model_df.shape)

print("\nBaseline columns:")
print(baseline_df.columns.tolist())

print("\nMacro columns:")
print(macro_df.columns.tolist())

print("\nFull model columns:")
print(full_model_df.columns.tolist())

In [ ]:
# =========================================================
# Convert all numeric columns to float64
# =========================================================

def convert_to_float(df):
    for col in df.columns:
        if col != "date":
            df[col] = df[col].astype(float)
    return df

baseline_df = convert_to_float(baseline_df)
macro_df = convert_to_float(macro_df)
full_model_df = convert_to_float(full_model_df)


In [ ]:
# =========================================================
#  Saving dataframe to desk 
# =========================================================


# Create folder if it doesn't exist
os.makedirs("data", exist_ok=True)

# Save datasets
baseline_df.to_parquet("data/baseline_df.parquet", index=False)
macro_df.to_parquet("data/macro_df.parquet", index=False)
full_model_df.to_parquet("data/full_model_df.parquet", index=False)

master_df_impute_eu.to_parquet("data/master_df_impute_eu.parquet", index=False)
